In [ ]:
import os
from pathlib import Path

# Ensure working directory is repo root regardless of how this notebook was opened
_cwd = Path(os.getcwd())
if _cwd.name in ("notebooks", "supplementary"):
    os.chdir(_cwd.parent.parent if _cwd.name == "supplementary" else _cwd.parent)


# QuantumEdge - Mandatory MNIST QRC Expressivity Benchmark

This notebook implements the common Phase 3 MNIST benchmark using the same core design choices as the financial system: fixed Ising dynamics, angle encoding, dual reservoirs, Pauli `Z` and nearest-neighbour `ZZ` readout, and a classical linear head.

**Official evidence rule:** `results/mnist_qrc_metrics.csv` is created only when the canonical MNIST dataset is loaded. The optional sklearn digits run is a code smoke test and is written to a different filename; it must not be reported as MNIST.

The official run evaluates 5, 10, and 15 qubits on a balanced, seeded subset to keep a judge-side rerun practical. No QPU is required.


## Run instructions

1. Preferred: include `data/mnist.npz` in Keras format (`x_train`, `y_train`, `x_test`, `y_test`).
2. Otherwise, leave `ALLOW_MNIST_DOWNLOAD=True`; Keras will download and cache the canonical dataset.
3. Set `RUN_PROFILE="FULL"` and choose **Restart & Run All**.
4. Confirm `results/mnist_qrc_metrics.csv`, `results/mnist_confusion_matrix_q15.csv`, and `figures/mnist_accuracy_by_qubits.png` exist.
5. Run Notebook 3. It rejects the smoke-test file as official evidence.


In [ ]:
from pathlib import Path
import os

_START_DIR = Path.cwd().resolve()

if _START_DIR.name == "notebooks":
    PROJECT_ROOT = _START_DIR.parent
elif (_START_DIR / "notebooks").exists():
    PROJECT_ROOT = _START_DIR
else:
    # Preserve the current directory for development copies placed at root.
    PROJECT_ROOT = _START_DIR

os.chdir(PROJECT_ROOT)
print("QuantumEdge project root:", PROJECT_ROOT)
print("Working directory:", Path.cwd().resolve())


In [ ]:
import os
import time
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

SEED = 7
rng = np.random.default_rng(SEED)
for folder in ["data", "results", "figures"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

RUN_PROFILE = "FULL"   # FULL is required for submission; QUICK is a development smoke test.
ALLOW_MNIST_DOWNLOAD = True
ALLOW_DIGITS_SMOKE_TEST = True
QUBIT_COUNTS = [5, 10, 15]
TRAIN_PER_CLASS = 10 if RUN_PROFILE == "FULL" else 2
TEST_PER_CLASS = 5 if RUN_PROFILE == "FULL" else 1

print("Profile:", RUN_PROFILE)
print("Qubits:", QUBIT_COUNTS)
print("Balanced subset per class:", TRAIN_PER_CLASS, "train /", TEST_PER_CLASS, "test")


## 1. Load canonical MNIST


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_mnist():
    local = Path("data/mnist.npz")
    if local.exists():
        with np.load(local) as data:
            required = {"x_train", "y_train", "x_test", "y_test"}
            if not required.issubset(data.files):
                raise ValueError("data/mnist.npz does not contain the four canonical Keras arrays.")
            return data["x_train"], data["y_train"], data["x_test"], data["y_test"], f"local {local}", True
    if ALLOW_MNIST_DOWNLOAD:
        try:
            from urllib.request import urlretrieve
            url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
            urlretrieve(url, local)
            with np.load(local) as data:
                return data["x_train"], data["y_train"], data["x_test"], data["y_test"], url, True
        except Exception as direct_exc:
            print("Direct canonical MNIST download failed:", type(direct_exc).__name__, direct_exc)
            try:
                from keras.datasets import mnist
                (x_train, y_train), (x_test, y_test) = mnist.load_data()
                np.savez_compressed(local, x_train=x_train, y_train=y_train, x_test=x_test, y_test=y_test)
                return x_train, y_train, x_test, y_test, "keras.datasets.mnist", True
            except Exception as exc:
                print("Keras MNIST download failed:", type(exc).__name__, exc)
    if ALLOW_DIGITS_SMOKE_TEST:
        from sklearn.datasets import load_digits
        digits = load_digits()
        images = digits.images
        labels = digits.target
        split = int(0.8 * len(images))
        print("WARNING: using sklearn digits only as a smoke test. This is NOT MNIST evidence.")
        return images[:split], labels[:split], images[split:], labels[split:], "sklearn digits smoke test", False
    raise RuntimeError("Canonical MNIST is unavailable and smoke-test fallback is disabled.")

x_train_all, y_train_all, x_test_all, y_test_all, DATA_SOURCE, IS_CANONICAL_MNIST = load_mnist()
print("Source:", DATA_SOURCE)
print("Canonical MNIST:", IS_CANONICAL_MNIST)
print("Shapes:", x_train_all.shape, x_test_all.shape)


## 2. Balanced subset and 4x4 pooled image encoding


In [ ]:
def balanced_subset(images, labels, per_class, seed):
    local_rng = np.random.default_rng(seed)
    chosen = []
    for digit in range(10):
        candidates = np.flatnonzero(labels == digit)
        if len(candidates) < per_class:
            raise ValueError(f"Class {digit} has only {len(candidates)} samples; {per_class} requested.")
        chosen.extend(local_rng.choice(candidates, per_class, replace=False))
    chosen = np.asarray(chosen)
    local_rng.shuffle(chosen)
    return images[chosen], labels[chosen]


def pool_to_4x4(images):
    images = images.astype(np.float64)
    if images.shape[1:] == (28, 28):
        # Crop to 24x24, then average 6x6 blocks.
        cropped = images[:, 2:26, 2:26]
        pooled = cropped.reshape(len(images), 4, 6, 4, 6).mean(axis=(2, 4))
        scale = 255.0
    elif images.shape[1:] == (8, 8):
        pooled = images.reshape(len(images), 4, 2, 4, 2).mean(axis=(2, 4))
        scale = 16.0
    else:
        raise ValueError(f"Unsupported image shape: {images.shape[1:]}")
    return np.clip(pooled.reshape(len(images), 16) / scale, 0.0, 1.0)

x_train, y_train = balanced_subset(x_train_all, y_train_all, TRAIN_PER_CLASS, SEED)
x_test, y_test = balanced_subset(x_test_all, y_test_all, TEST_PER_CLASS, SEED + 1)
pooled_train = pool_to_4x4(x_train)
pooled_test = pool_to_4x4(x_test)
print("Official subset:" if IS_CANONICAL_MNIST else "Smoke subset:", pooled_train.shape, pooled_test.shape)


## 3. Pure NumPy dual-timescale quantum reservoir


In [ ]:
def ry(theta):
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    return np.array([[c, -s], [s, c]], dtype=np.complex128)


def rx(theta):
    c, s = np.cos(theta / 2), -1j * np.sin(theta / 2)
    return np.array([[c, s], [s, c]], dtype=np.complex128)


def rzz(theta):
    return np.diag([
        np.exp(-0.5j * theta), np.exp(0.5j * theta),
        np.exp(0.5j * theta), np.exp(-0.5j * theta),
    ]).astype(np.complex128)


def zero_mps(n):
    tensors = []
    for _ in range(n):
        tensor = np.zeros((1, 2, 1), dtype=np.complex128)
        tensor[0, 0, 0] = 1.0
        tensors.append(tensor)
    return tensors


def apply_one_mps(mps, gate, qubit):
    mps[qubit] = np.einsum("ij,ajb->aib", gate, mps[qubit], optimize=True)


def apply_two_mps(mps, gate, left_qubit, max_bond=None):
    A, B = mps[left_qubit], mps[left_qubit + 1]
    left_dim, _, middle = A.shape
    middle_b, _, right_dim = B.shape
    if middle != middle_b:
        raise ValueError("MPS bond mismatch")
    theta = np.einsum("aib,bjc->aijc", A, B, optimize=True)
    theta = theta.reshape(left_dim, 4, right_dim)
    theta = np.einsum("pq,aqc->apc", gate, theta, optimize=True)
    matrix = theta.reshape(left_dim * 2, 2 * right_dim)
    U, singular, Vh = np.linalg.svd(matrix, full_matrices=False)
    keep = len(singular) if max_bond is None else min(max_bond, len(singular))
    U, singular, Vh = U[:, :keep], singular[:keep], Vh[:keep]
    mps[left_qubit] = U.reshape(left_dim, 2, keep)
    mps[left_qubit + 1] = (singular[:, None] * Vh).reshape(keep, 2, right_dim)


def mps_to_statevector(mps):
    state = mps[0][0]  # physical, right bond
    for tensor in mps[1:]:
        state = np.tensordot(state, tensor, axes=([-1], [0]))
    return state.reshape(-1)


def qrc_state(angles, n, J, h, repetitions):
    mps = zero_mps(n)
    for _ in range(repetitions):
        for q in range(n):
            apply_one_mps(mps, ry(float(angles[q % len(angles)])), q)
        gate = rzz(2 * J)
        for q in range(n - 1):
            apply_two_mps(mps, gate, q, max_bond=4)
        for q in range(n):
            apply_one_mps(mps, rx(2 * h), q)
    return mps_to_statevector(mps)


_SIGN_CACHE = {}


def z_zz_features(state, n):
    probabilities = np.abs(state) ** 2
    probabilities = probabilities / probabilities.sum()
    if n not in _SIGN_CACHE:
        indices = np.arange(2 ** n, dtype=np.uint32)
        singles = np.stack([
            1.0 - 2.0 * ((indices >> (n - 1 - q)) & 1)
            for q in range(n)
        ]).astype(np.float64)
        pairs = singles[:-1] * singles[1:]
        _SIGN_CACHE[n] = (singles, pairs)
    singles, pairs = _SIGN_CACHE[n]
    return np.concatenate([singles @ probabilities, pairs @ probabilities])


def project_pixels(pixels, n, seed):
    local_rng = np.random.default_rng(seed + n)
    projection = local_rng.normal(size=(pixels.shape[1], n)) / np.sqrt(pixels.shape[1])
    projected = pixels @ projection
    scale = np.quantile(np.abs(projected), 0.99) + 1e-12
    return np.clip(projected / scale, -1, 1) * (np.pi / 2)


def extract_dual_features(pixels, n, seed):
    cache = Path(f"data/mnist_qrc_features_{'mnist' if IS_CANONICAL_MNIST else 'digits'}_q{n}_{len(pixels)}.npz")
    signature = hashlib.sha256(np.ascontiguousarray(pixels).view(np.uint8)).hexdigest()
    if cache.exists():
        saved = np.load(cache, allow_pickle=False)
        if str(saved["signature"]) == signature:
            return saved["features"], float(saved["wall_seconds"])
    angles = project_pixels(pixels, n, seed)
    rows = []
    start = time.time()
    for row_number, row in enumerate(angles, start=1):
        short_state = qrc_state(row, n, J=0.3, h=1.0, repetitions=1)
        long_state = qrc_state(row, n, J=1.2, h=0.4, repetitions=1)
        rows.append(np.concatenate([z_zz_features(short_state, n), z_zz_features(long_state, n)]))
        if row_number % 25 == 0:
            print(f"q={n}: {row_number}/{len(angles)} images")
    elapsed = time.time() - start
    features = np.asarray(rows)
    np.savez_compressed(cache, features=features, wall_seconds=elapsed, signature=signature)
    return features, elapsed


## 4. Qubit scaling, classical control, and exported evidence


In [ ]:
all_pixels = np.vstack([pooled_train, pooled_test])
metrics = []
confusions = {}
for n in QUBIT_COUNTS:
    features, feature_seconds = extract_dual_features(all_pixels, n, SEED)
    train_features = features[:len(pooled_train)]
    test_features = features[len(pooled_train):]
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=4000, random_state=SEED))
    fit_start = time.time()
    model.fit(train_features, y_train)
    predictions = model.predict(test_features)
    fit_seconds = time.time() - fit_start
    metrics.append(dict(
        dataset="MNIST" if IS_CANONICAL_MNIST else "sklearn_digits_smoke_test",
        qubits=n,
        reservoir_features=int(features.shape[1]),
        train_samples=int(len(y_train)),
        test_samples=int(len(y_test)),
        accuracy=float(accuracy_score(y_test, predictions)),
        balanced_accuracy=float(balanced_accuracy_score(y_test, predictions)),
        macro_f1=float(f1_score(y_test, predictions, average="macro")),
        feature_wall_seconds=float(feature_seconds),
        readout_wall_seconds=float(fit_seconds),
        seed=SEED,
    ))
    confusions[n] = confusion_matrix(y_test, predictions, labels=np.arange(10))

# Classical control on the identical pooled pixels.
classical = make_pipeline(StandardScaler(), LogisticRegression(max_iter=4000, random_state=SEED))
start = time.time()
classical.fit(pooled_train, y_train)
classical_pred = classical.predict(pooled_test)
metrics.append(dict(
    dataset="MNIST" if IS_CANONICAL_MNIST else "sklearn_digits_smoke_test",
    qubits=0,
    reservoir_features=int(pooled_train.shape[1]),
    train_samples=int(len(y_train)),
    test_samples=int(len(y_test)),
    accuracy=float(accuracy_score(y_test, classical_pred)),
    balanced_accuracy=float(balanced_accuracy_score(y_test, classical_pred)),
    macro_f1=float(f1_score(y_test, classical_pred, average="macro")),
    feature_wall_seconds=0.0,
    readout_wall_seconds=float(time.time() - start),
    seed=SEED,
))

mnist_metrics = pd.DataFrame(metrics)
output_metrics = Path("results/mnist_qrc_metrics.csv" if IS_CANONICAL_MNIST else "results/mnist_smoke_test_metrics.csv")
mnist_metrics.to_csv(output_metrics, index=False)
for n, matrix in confusions.items():
    target = Path(f"results/{'mnist' if IS_CANONICAL_MNIST else 'digits_smoke'}_confusion_matrix_q{n}.csv")
    pd.DataFrame(matrix, index=np.arange(10), columns=np.arange(10)).to_csv(target)

manifest = {
    "generated_utc": pd.Timestamp.utcnow().isoformat(),
    "canonical_mnist": IS_CANONICAL_MNIST,
    "data_source": DATA_SOURCE,
    "profile": RUN_PROFILE,
    "qubits": QUBIT_COUNTS,
    "train_per_class": TRAIN_PER_CLASS,
    "test_per_class": TEST_PER_CLASS,
    "result_file": str(output_metrics),
}
Path("results/mnist_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

display(mnist_metrics)
print("Saved:", output_metrics)
if not IS_CANONICAL_MNIST:
    print("STOP: the smoke-test output does not satisfy the mandatory MNIST requirement.")


In [ ]:
plot_data = mnist_metrics.copy()
plot_data["label"] = plot_data["qubits"].map(lambda value: "Pooled logistic" if value == 0 else f"QRC {value}q")
fig, ax = plt.subplots(figsize=(7.2, 4.1))
ax.bar(plot_data["label"], plot_data["accuracy"])
ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy")
ax.set_title("MNIST QRC expressivity across qubit counts" if IS_CANONICAL_MNIST else "Digits smoke test - not official MNIST")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
figure_path = Path("figures/mnist_accuracy_by_qubits.png" if IS_CANONICAL_MNIST else "figures/digits_smoke_accuracy.png")
fig.savefig(figure_path, dpi=180)
plt.show()
print("Saved:", figure_path)


## Interpretation guardrails

- This benchmark tests feature separability, not financial forecasting.
- The pooled-logistic control establishes whether the selected subset is already easy without the reservoir.
- Qubit scaling is empirical; larger reservoirs are not assumed to improve accuracy monotonically.
- A small balanced subset is used for reproducibility and cost control. The subset size, seed, runtime, and data source are exported.
- Only a run with `canonical_mnist=true` in `results/mnist_manifest.json` may be cited as the Phase 3 MNIST result.
